# Create pseudo-label training file

This notebook converts a nuScenes camera-only detection JSON file into a mmdetection3d-friendly pickle file.
It keeps only samples from the provided info pickle, applies a configurable confidence threshold, and writes pseudo ground-truth boxes in LiDAR format.

In [6]:
import copy
import json
import pickle
from pathlib import Path

import numpy as np
from nuscenes.utils.data_classes import Box as NuScenesBox
from pyquaternion import Quaternion

# Input files
results_camera_only_bounding_boxes_path = Path(
    '/home/mingdayang/mmdetection3d/outputs/inference/camera_pseudo_experiment/20-04-2026-1_unibev_val_C_full_nuscenes_train_inference/pts_bbox/results_nusc.json',
)
pickle_file_nuscenes_temporal_train = Path(
    '/home/mingdayang/mmdetection3d/data/nuscenes/mmdet3d_bevformer/nuscenes_infos_temporal_train.pkl',
)
custom_annotation_dir = Path('/home/mingdayang/mmdetection3d/data/nuscenes/mmdet3d_bevformer/nuscenes_annotation_files_custom')
custom_annotation_dir.mkdir(parents=True, exist_ok=True)

# User-editable settings
confidence_threshold = 0.4
keep_empty_samples = False

# Output file
threshold_tag = str(confidence_threshold).replace('.', 'p')
output_pseudo_pickle_path = custom_annotation_dir / (
    f'{pickle_file_nuscenes_temporal_train.stem}_pseudo_thr{threshold_tag}.pkl'
)

allowed_nuscenes_classes = {
    'car', 'truck', 'trailer', 'bus', 'construction_vehicle',
    'bicycle', 'motorcycle', 'pedestrian', 'traffic_cone', 'barrier'
}

print(output_pseudo_pickle_path)

/home/mingdayang/mmdetection3d/data/nuscenes/mmdet3d_bevformer/nuscenes_annotation_files_custom/nuscenes_infos_temporal_train_pseudo_thr0p4.pkl


In [3]:
def load_info_pickle(info_path):
    with info_path.open('rb') as f:
        data = pickle.load(f)

    if 'infos' in data and 'metadata' in data:
        info_key = 'infos'
        meta_key = 'metadata'
    elif 'data_list' in data and 'metainfo' in data:
        info_key = 'data_list'
        meta_key = 'metainfo'
    else:
        raise KeyError(
            'Unsupported info pickle format. Expected either {infos, metadata} or {data_list, metainfo}.',
        )

    return data, data[info_key], data[meta_key], info_key, meta_key


def load_results_json(results_path):
    with results_path.open('r') as f:
        results = json.load(f)

    if 'results' not in results:
        raise KeyError("The JSON file must contain a top-level 'results' key.")

    return results['results']


def transform_prediction_to_lidar(pred, info):
    box = NuScenesBox(
        center=np.asarray(pred['translation'], dtype=np.float32),
        size=np.asarray(pred['size'], dtype=np.float32),
        orientation=Quaternion(pred['rotation']),
        velocity=np.asarray(pred.get('velocity', [0.0, 0.0, 0.0]), dtype=np.float32),
        name=pred.get('detection_name'),
        score=float(pred.get('detection_score', pred.get('score', 0.0))),
    )

    box.translate(-np.asarray(info['ego2global_translation'], dtype=np.float32))
    box.rotate(Quaternion(info['ego2global_rotation']).inverse)
    box.translate(-np.asarray(info['lidar2ego_translation'], dtype=np.float32))
    box.rotate(Quaternion(info['lidar2ego_rotation']).inverse)

    center = box.center.astype(np.float32)
    size = box.wlh.astype(np.float32)
    yaw = float(box.orientation.yaw_pitch_roll[0])

    velocity_global = np.asarray(pred.get('velocity', [0.0, 0.0]), dtype=np.float32)
    if velocity_global.shape[0] >= 2:
        velocity_3d = np.array([velocity_global[0], velocity_global[1], 0.0], dtype=np.float32)
        e2g_r = Quaternion(info['ego2global_rotation']).rotation_matrix
        l2e_r = Quaternion(info['lidar2ego_rotation']).rotation_matrix
        velocity_lidar = velocity_3d @ np.linalg.inv(e2g_r).T @ np.linalg.inv(l2e_r).T
        velocity_lidar = velocity_lidar[:2].astype(np.float32)
    else:
        velocity_lidar = np.zeros(2, dtype=np.float32)

    lidar_box = np.array(
        [
            center[0],
            center[1],
            center[2],
            size[0],
            size[1],
            size[2],
            -yaw - np.pi / 2,
        ],
        dtype=np.float32,
    )
    return lidar_box, velocity_lidar


def build_pseudo_infos(results_path, info_path, score_threshold=0.4, keep_empty=True):
    source_data, source_infos, source_meta, info_key, meta_key = load_info_pickle(info_path)
    prediction_dict = load_results_json(results_path)
    subset_tokens = {info['token'] for info in source_infos}
    result_tokens = set(prediction_dict.keys())

    pseudo_infos = []
    total_prediction_entries = 0
    kept_prediction_entries = 0
    samples_with_kept_boxes = 0

    for info in source_infos:
        pseudo_info = copy.deepcopy(info)
        predictions = prediction_dict.get(info['token'], [])

        gt_boxes = []
        gt_names = []
        gt_velocity = []
        num_lidar_pts = []
        num_radar_pts = []
        valid_flag = []

        for pred in predictions:
            score = float(pred.get('detection_score', pred.get('score', 0.0)))
            if score < score_threshold:
                continue

            name = pred.get('detection_name')
            if name not in allowed_nuscenes_classes:
                continue

            box_lidar, vel_lidar = transform_prediction_to_lidar(pred, info)
            gt_boxes.append(box_lidar)
            gt_names.append(name)
            gt_velocity.append(vel_lidar)
            num_lidar_pts.append(1)
            num_radar_pts.append(0)
            valid_flag.append(True)

        total_prediction_entries += len(predictions)
        kept_prediction_entries += len(gt_boxes)
        if len(gt_boxes) > 0:
            samples_with_kept_boxes += 1

        if gt_boxes:
            pseudo_info['gt_boxes'] = np.asarray(gt_boxes, dtype=np.float32)
            pseudo_info['gt_names'] = np.asarray(gt_names)
            pseudo_info['gt_velocity'] = np.asarray(gt_velocity, dtype=np.float32)
            pseudo_info['num_lidar_pts'] = np.asarray(num_lidar_pts, dtype=np.int64)
            pseudo_info['num_radar_pts'] = np.asarray(num_radar_pts, dtype=np.int64)
            pseudo_info['valid_flag'] = np.asarray(valid_flag, dtype=bool)
        else:
            pseudo_info['gt_boxes'] = np.zeros((0, 7), dtype=np.float32)
            pseudo_info['gt_names'] = np.asarray([], dtype=object)
            pseudo_info['gt_velocity'] = np.zeros((0, 2), dtype=np.float32)
            pseudo_info['num_lidar_pts'] = np.zeros((0,), dtype=np.int64)
            pseudo_info['num_radar_pts'] = np.zeros((0,), dtype=np.int64)
            pseudo_info['valid_flag'] = np.zeros((0,), dtype=bool)

        if keep_empty or len(gt_boxes) > 0:
            pseudo_infos.append(pseudo_info)

    output_data = {info_key: pseudo_infos, meta_key: source_meta}
    stats = {
        'subset_input_samples': len(source_infos),
        'subset_output_samples': len(pseudo_infos),
        'result_json_samples': len(result_tokens),
        'subset_samples_with_predictions': len(subset_tokens & result_tokens),
        'total_prediction_entries_in_subset': total_prediction_entries,
        'kept_prediction_entries': kept_prediction_entries,
        'samples_with_at_least_one_box': samples_with_kept_boxes,
        'info_key': info_key,
        'meta_key': meta_key,
    }
    return output_data, stats

In [4]:
# Hotfix: ensure prediction velocity is always 3D before NuScenesBox rotations
def transform_prediction_to_lidar(pred, info):
    velocity_raw = pred.get('velocity', [0.0, 0.0])
    velocity_arr = np.asarray(velocity_raw, dtype=np.float32).reshape(-1)

    if velocity_arr.shape[0] >= 3:
        velocity_3d_global = velocity_arr[:3]
    elif velocity_arr.shape[0] == 2:
        velocity_3d_global = np.array([velocity_arr[0], velocity_arr[1], 0.0], dtype=np.float32)
    else:
        velocity_3d_global = np.zeros(3, dtype=np.float32)

    box = NuScenesBox(
        center=np.asarray(pred['translation'], dtype=np.float32),
        size=np.asarray(pred['size'], dtype=np.float32),
        orientation=Quaternion(pred['rotation']),
        velocity=velocity_3d_global,
        name=pred.get('detection_name'),
        score=float(pred.get('detection_score', pred.get('score', 0.0))),
    )

    box.translate(-np.asarray(info['ego2global_translation'], dtype=np.float32))
    box.rotate(Quaternion(info['ego2global_rotation']).inverse)
    box.translate(-np.asarray(info['lidar2ego_translation'], dtype=np.float32))
    box.rotate(Quaternion(info['lidar2ego_rotation']).inverse)

    center = box.center.astype(np.float32)
    size = box.wlh.astype(np.float32)
    yaw = float(box.orientation.yaw_pitch_roll[0])

    velocity_lidar = np.asarray(box.velocity[:2], dtype=np.float32)

    lidar_box = np.array(
        [
            center[0],
            center[1],
            center[2],
            size[0],
            size[1],
            size[2],
            -yaw - np.pi / 2,
        ],
        dtype=np.float32,
    )
    return lidar_box, velocity_lidar

In [7]:
pseudo_data, conversion_stats = build_pseudo_infos(
    results_camera_only_bounding_boxes_path,
    pickle_file_nuscenes_temporal_train,
    score_threshold=confidence_threshold,
    keep_empty=keep_empty_samples,
 )

with output_pseudo_pickle_path.open('wb') as f:
    pickle.dump(pseudo_data, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved pseudo-label pickle to: {output_pseudo_pickle_path}')
for key, value in conversion_stats.items():
    print(f'{key}: {value}')

Saved pseudo-label pickle to: /home/mingdayang/mmdetection3d/data/nuscenes/mmdet3d_bevformer/nuscenes_annotation_files_custom/nuscenes_infos_temporal_train_pseudo_thr0p4.pkl
subset_input_samples: 28130
subset_output_samples: 27963
result_json_samples: 28130
subset_samples_with_predictions: 28130
total_prediction_entries_in_subset: 6228561
kept_prediction_entries: 565508
samples_with_at_least_one_box: 27963
info_key: infos
meta_key: metadata


In [8]:
prediction_dict = load_results_json(results_camera_only_bounding_boxes_path)
print(len(prediction_dict), 'samples in results JSON')

28130 samples in results JSON


In [ ]:
with output_pseudo_pickle_path.open('rb') as f:
    loaded_output = pickle.load(f)

if 'infos' in loaded_output:
    output_info_key = 'infos'
elif 'data_list' in loaded_output:
    output_info_key = 'data_list'
else:
    raise KeyError('The saved pickle does not contain infos/data_list.')

print(f'Top-level keys: {list(loaded_output.keys())}')
print(f'Number of pseudo-labeled samples: {len(loaded_output[output_info_key])}')

first_sample = loaded_output[output_info_key][0]
print(f'First sample token: {first_sample["token"]}')
print(f'First sample pseudo boxes: {first_sample["gt_boxes"].shape[0]}')
if first_sample['gt_boxes'].shape[0] > 0:
    print('First pseudo box:')
    print(first_sample['gt_boxes'][0])

## Usage notes

- Change `confidence_threshold` in the first code cell to control pseudo-label filtering.
- Keep `keep_empty_samples = True` if you want the output pickle to preserve the full subset from the input info file.
- Use the saved pickle path as the `ann_file` for mmdetection3d training.